# RAG Pipeline Walkthrough

This notebook follows Phase 6 step by step. The goal is to understand each RAG concept before moving to the next implementation layer.

Current completed steps:

1. Inspect available knowledge sources
2. Load documents into a common shape
3. Normalize document text
4. Chunk documents with LangChain's RecursiveCharacterTextSplitter
5. Preview embeddings in memory
6. Full embedding / indexing / Chroma store
7. Retrieve relevant chunks
8. Generate preview answers with citations
9. Call the backend chat API from Python
10. Design and implement the LangGraph support agent
11. Wire the backend `/chat` endpoint to the LangGraph workflow
12. Validate support-agent behavior with repeatable cases

Upcoming steps:

13. Connect the frontend support chat UI

## Setup

Run this notebook from the project root or from the `rag_pipeline/` folder. The cell below makes imports work either way.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
project_root = cwd if (cwd / "rag_pipeline").exists() else cwd.parent
rag_dir = project_root / "rag_pipeline"
backend_dir = project_root / "backend"

if str(rag_dir) not in sys.path:
    sys.path.append(str(rag_dir))
if str(backend_dir) not in sys.path:
    sys.path.append(str(backend_dir))

project_root

WindowsPath('c:/Users/uday0/Documents/GenAIAcademy/Agentic-AI/CSAgent')

### Dependency Check

If imports fail later, this cell usually tells you why. The notebook should use the backend virtual environment, or it should be launched from `backend/` with `uv` so packages like `langchain_chroma` are available.

In [2]:
import sys

print("Python executable:", sys.executable)

required_packages = [
    "langchain_chroma",
    "langchain_openai",
    "langchain_text_splitters",
    "chromadb",
    "langgraph",
]

for package_name in required_packages:
    try:
        __import__(package_name)
        print(f"ok: {package_name}")
    except ModuleNotFoundError as error:
        print(f"missing: {package_name} ({error})")

Python executable: c:\Users\uday0\Documents\GenAIAcademy\Agentic-AI\CSAgent\backend\.venv\Scripts\python.exe
ok: langchain_chroma
ok: langchain_openai
ok: langchain_text_splitters
ok: chromadb
ok: langgraph


## Step 1: Inspect Knowledge Sources

Before building retrieval, we need to know what knowledge exists.

RAG quality depends on the available documents. If the right information is missing here, embeddings and retrieval cannot magically recover it later.

In [ ]:
import json
from collections import Counter

data_root = project_root / "data"
catalog_path = data_root / "catalog" / "products.json"
categories_path = data_root / "catalog" / "categories.json"
reviews_path = data_root / "reviews" / "product_reviews.json"
kb_root = data_root / "knowledge_base"
tickets_path = data_root / "support" / "tickets.json"

products = json.loads(catalog_path.read_text(encoding="utf-8"))
categories = json.loads(categories_path.read_text(encoding="utf-8"))
reviews_by_product = json.loads(reviews_path.read_text(encoding="utf-8"))
tickets = json.loads(tickets_path.read_text(encoding="utf-8"))

product_ids = {product["id"] for product in products}
review_counts = {product_id: len(reviews_by_product.get(product_id, [])) for product_id in product_ids}
ticket_issue_counts = Counter(ticket["issueType"] for ticket in tickets)

inventory = {
    "products": len(products),
    "categories": len(categories),
    "products_with_reviews": sum(count > 0 for count in review_counts.values()),
    "products_with_6_plus_reviews": sum(count >= 6 for count in review_counts.values()),
    "total_review_samples": sum(review_counts.values()),
    "store_policy_docs": len(list((kb_root / "store_policies").rglob("*.md"))),
    "product_kb_docs": len(list((kb_root / "products").rglob("*.md"))),
    "total_kb_markdown_docs": len(list(kb_root.rglob("*.md"))),
    "support_tickets": len(tickets),
    "ticket_issue_counts": dict(sorted(ticket_issue_counts.items())),
}

inventory

## Step 2: Load Documents

The loader converts every source into one common shape:

```python
KnowledgeDocument(
    page_content="searchable text",
    metadata={...}
)
```

`page_content` is what will eventually be chunked and embedded. `metadata` lets us filter, cite, and debug retrieved context.

In [ ]:
from document_loader import compact_text, load_all_documents

documents = load_all_documents()
len(documents)

In [ ]:
source_counts = Counter(document.metadata["source_type"] for document in documents)
doc_type_counts = Counter(document.metadata.get("doc_type", "unknown") for document in documents)

source_counts, doc_type_counts

In [ ]:
def show_sample_by_doc_type(doc_type: str, limit: int = 500):
    for document in documents:
        if document.metadata.get("doc_type") == doc_type:
            print("Metadata:")
            print(document.metadata)
            print("\nContent preview:")
            print(compact_text(document.page_content)[:limit])
            return
    print(f"No document found for doc_type={doc_type!r}")

show_sample_by_doc_type("product_faq")

## Step 3: Normalize Text

Loading gets data into memory. Normalization makes the text more consistent and retrieval-friendly.

The normalizer currently:

- compacts whitespace
- standardizes boolean text like `Verified purchase: True` to `Verified purchase: yes`
- adds context from metadata
- preserves metadata
- marks documents with `normalized: True`

In [ ]:
from text_normalizer import load_normalized_documents

normalized_documents = load_normalized_documents()

len(documents), len(normalized_documents)

In [ ]:
normalized_counts = Counter(document.metadata.get("doc_type", "unknown") for document in normalized_documents)
normalized_counts

In [ ]:
def show_normalized_sample(doc_type: str, limit: int = 700):
    for document in normalized_documents:
        if document.metadata.get("doc_type") == doc_type:
            print("Metadata:")
            print(document.metadata)
            print("\nNormalized content preview:")
            print(compact_text(document.page_content)[:limit])
            return
    print(f"No document found for doc_type={doc_type!r}")

show_normalized_sample("customer_review")

## Step 4: Chunk Documents

Chunking breaks longer documents into retrieval-sized passages.

We use LangChain's `RecursiveCharacterTextSplitter` because it tries to split on natural boundaries first: paragraphs, then lines, then sentences, then words. That gives us chunks that are more readable than cutting every 900 characters blindly.

Current settings:

- `chunk_size = 900`: large enough to keep product, policy, and troubleshooting context together
- `chunk_overlap = 150`: repeats a little context between neighboring chunks so answers do not lose important setup from the previous chunk
- separators: paragraph, line, sentence, word, then character fallback

In [ ]:
from chunking import DEFAULT_CHUNK_OVERLAP, DEFAULT_CHUNK_SIZE, load_chunked_documents

chunks = load_chunked_documents()

len(normalized_documents), len(chunks), DEFAULT_CHUNK_SIZE, DEFAULT_CHUNK_OVERLAP

In [ ]:
chunk_counts = Counter(chunk.metadata.get("doc_type", "unknown") for chunk in chunks)
chunk_counts

In [ ]:
def show_chunk_sample(doc_type: str, limit: int = 700):
    for chunk in chunks:
        if chunk.metadata.get("doc_type") == doc_type:
            print("Metadata:")
            print(chunk.metadata)
            print("\nChunk content preview:")
            print(compact_text(chunk.page_content)[:limit])
            return
    print(f"No chunk found for doc_type={doc_type!r}")

show_chunk_sample("troubleshooting")

After chunking, the document count increases because one source document can become multiple retrieval passages.

In the command-line preview, 479 normalized documents became 907 chunks. Each chunk keeps the original metadata and adds chunk-specific fields like `chunk_id`, `chunk_index`, `chunk_count`, `chunk_size`, and `chunk_overlap`.

## Step 5: Preview Embeddings In Memory

Embeddings convert each text chunk into a numeric vector. Retrieval works by comparing the query vector to chunk vectors and finding the nearest matches.

This step is only for learning and inspection. It embeds a tiny sample of chunks in memory so you can see what an embedding record looks like. These preview vectors are not saved to Chroma and are lost when the notebook kernel stops.

The current persisted Chroma vector store was built using Nebius embeddings, so the notebook uses `nebius` for the preview too. Chroma is the vector database; Nebius is only the embedding provider that creates the vectors stored in Chroma.

The output shape is:

```python
EmbeddingRecord(
    id="stable chunk id",
    text="chunk text",
    embedding=[...numbers...],
    metadata={...}
)
```

In [ ]:
from embeddings import get_embedding_model, embed_chunks

embedding_preview_provider = "nebius"
embedding_model = get_embedding_model(embedding_preview_provider)
embedding_model.name, embedding_model.dimension

In [ ]:
# Preview only: embed a small sample so the notebook stays fast.
# The full Chroma build in Step 6 embeds and persists all chunks.
preview_chunks = chunks[:5]
embedding_records = embed_chunks(preview_chunks, embedding_model=embedding_model)

len(preview_chunks), len(embedding_records), len(embedding_records[0].embedding)

In [ ]:
embedding_counts = Counter(record.metadata.get("doc_type", "unknown") for record in embedding_records)
embedding_counts

In [ ]:
sample_embedding = embedding_records[0]

print("ID:", sample_embedding.id)
print("Provider:", sample_embedding.metadata["embedding_provider"])
print("Dimension:", sample_embedding.metadata["embedding_dimension"])
print("Text preview:", compact_text(sample_embedding.text)[:400])
print("First 12 vector values:", sample_embedding.embedding[:12])

OpenAI embeddings understand similarity beyond exact words. For example, `refund`, `return`, and `money back` can land near each other in vector space even if the wording differs.

Semantic embeddings understand similarity beyond exact words. For example, `refund`, `return`, and `money back` can land near each other in vector space even if the wording differs.

The key point: Step 5 is a preview only. Step 6 is where all chunks are embedded again, indexed, and persisted in Chroma.

## Step 6: Full Embedding / Indexing / Chroma Store

This is the actual vector-store creation step. It embeds all chunks, stores those vectors locally in Chroma, and builds the index we will query in the next step.

Important connection: the Step 5 `embed_chunks(...)` call is only for previewing embeddings in memory. The actual Chroma build happens here with `rebuild_vector_store(...)`. During `vector_store.add_texts(...)`, LangChain/Chroma calls the embedding model again, stores the vectors, and indexes them in the local Chroma database.

The current persisted Chroma collection was built with the Nebius embedding provider. If you rebuild Chroma with another embedding provider, use that same embedding provider for retrieval later. A Chroma collection built with Nebius expects 4096-dimensional query embeddings, while the hashing fallback uses 384 dimensions.

In [ ]:
from vector_store import DEFAULT_COLLECTION_NAME, DEFAULT_VECTOR_STORE_DIR, rebuild_vector_store

chroma_embedding_provider = "nebius"

# This is the actual Chroma build step.
# It embeds chunk.page_content values and persists the indexed vectors to data/vector_store/chroma.
# Nebius works with the current project setup, but it may take several minutes for all 907 chunks.
vector_store = rebuild_vector_store(provider=chroma_embedding_provider, chunks=chunks)

DEFAULT_COLLECTION_NAME, DEFAULT_VECTOR_STORE_DIR

If you do not want to rebuild the vector store from the notebook, skip the cell above and use the already-persisted Chroma database. The current local database was already built with Nebius embeddings.

Equivalent command-line build:

```powershell
cd backend
uv run python ../rag_pipeline/build_vector_store.py --provider nebius
```

## Step 7: Retrieve Relevant Chunks

Retrieval is separate from vector-store creation. Now that Chroma has an index, we can embed a user question with the same embedding provider and ask Chroma for nearby chunks.

The retriever currently combines two signals:

- vector similarity from Chroma
- lightweight lexical overlap from chunk text and metadata

This hybrid ranking helps policy and product-specific words stay visible while still allowing semantic matches.

In [ ]:
from retrieval import retrieve_relevant_chunks

# This must match the embedding provider used to build the persisted Chroma index.
# The current local Chroma index was built with Nebius embeddings.
retrieval_embedding_provider = "nebius"
sample_question = "What is the return policy for opened electronics?"

retrieval_results = retrieve_relevant_chunks(sample_question, provider=retrieval_embedding_provider)
len(retrieval_results)

In [ ]:
from document_loader import compact_text

for index, result in enumerate(retrieval_results, start=1):
    metadata = result.document.metadata
    print(f"{index}. {metadata.get('doc_type')} | vector={result.vector_score:.4f} | lexical={result.lexical_score}")
    print("   source:", metadata.get("source_path"))
    print("   chunk:", metadata.get("chunk_id"))
    print("   text:", compact_text(result.document.page_content)[:240])
    print()

## Step 8: Answer Generation And Preview Answers

Answer generation is the final RAG layer: take the user question, retrieve relevant context, and produce an answer that points back to its sources.

For now, this project uses a deterministic extractive answer layer instead of calling a chat model. That keeps the pipeline usable while API quota and provider choices are still being sorted out. The answer layer:

- retrieves relevant chunks
- chooses the strongest matching sources
- extracts short support-context snippets
- adds numbered citations using source metadata
- asks for clarification when retrieved context is too weak

The reusable module is `rag_pipeline/answer_generation.py`, and the command-line preview is `rag_pipeline/preview_answers.py`.

In [ ]:
from answer_generation import generate_cited_answer

cited_answer = generate_cited_answer(
    question=sample_question,
    provider=retrieval_embedding_provider,
)

print(cited_answer.answer)

In [ ]:
for citation in cited_answer.citations:
    print(f"[{citation.id}] {citation.label}")
    print("   source:", citation.source_path)
    print("   doc type:", citation.doc_type)
    print("   product id:", citation.product_id)
    print("   chunk:", citation.chunk_id)
    print()

In [ ]:
preview_questions = [
    "What is the return policy for opened electronics?",
    "How do I troubleshoot a device that will not power on?",
    "Which product information mentions Bluetooth compatibility?",
]

for question in preview_questions:
    answer = generate_cited_answer(question=question, provider=retrieval_embedding_provider)
    print("Question:", question)
    print(answer.answer)
    print("Citations:")
    for citation in answer.citations:
        print(f"  [{citation.id}] {citation.label} | {citation.source_path}")
    print("-" * 80)

You can run the same preview outside the notebook with:

```powershell
cd backend
uv run python ../rag_pipeline/preview_answers.py --provider nebius
```

If you rebuild Chroma with hashing or OpenAI embeddings, pass the matching provider to both retrieval and answer preview commands. A Chroma collection built with Nebius expects 4096-dimensional query embeddings, while the hashing fallback uses 384 dimensions.

## Step 9: Backend Chat API Client

The backend exposes the LangGraph support-agent workflow through FastAPI at `POST /chat`.

This notebook client uses FastAPI's `TestClient`, so you do not need to start Uvicorn. It calls the API in process and returns the same JSON shape the frontend will receive later.

Request shape:

```json
{
  "question": "What is the return policy for opened electronics?",
  "provider": "nebius"
}
```

Response shape includes `answer`, `citations`, and `retrievedContext`. The public API shape is intentionally stable even though the internals now route through the graph.

In [ ]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

chat_response = client.post(
    "/chat",
    json={
        "question": "What is the return policy for opened clothes?",
        "provider": "nebius",
    },
)

chat_response.status_code

In [ ]:
chat_data = chat_response.json()

print(chat_data["answer"])
print("\nCitations:")
for citation in chat_data["citations"]:
    print(f"[{citation['id']}] {citation['label']} | {citation['sourcePath']}")

In [ ]:
print("Retrieved context candidates:")
for context in chat_data["retrievedContext"]:
    print(
        f"{context['rank']}. {context['docType']} | "
        f"vector={context['vectorScore']} | lexical={context['lexicalScore']} | "
        f"{context['sourcePath']}"
    )
    print("   ", context["snippet"][:220])

## Step 10: LangGraph Support Agent Design

Phase 7 keeps the working `/chat` API, but changes the internals from a single deterministic answer function into an explicit support-agent workflow.

The first design slice is the agent state. In LangGraph, each node reads and updates a shared state object. For this support flow, the state needs to carry the original question, embedding provider, detected intent, product hints, retrieved chunks, citations, answer text, context sufficiency, clarification status, and optional escalation metadata.

Current design file: `rag_pipeline/support_agent.py`.

The intended graph shape is:

```text
classify_intent -> retrieve_context -> assess_context
                                      -> answer
                                      -> clarify
                                      -> prepare_escalation
```

We are starting with deterministic routing and deterministic cited answers. LLM-based answer generation can be added later after a usable chat model/provider is selected.

In [ ]:
from support_agent import SupportIntent, initial_support_state

agent_state = initial_support_state(
    question="What is the return policy for opened electronics?",
    provider="nebius",
)

agent_state

Design notes for this slice:

- `SupportIntent` defines the stable labels that conditional graph edges will use.
- `SupportAgentState` is the mutable state shape passed between LangGraph nodes.
- `SupportAgentResult` is the final object that can be adapted back into the existing FastAPI `ChatResponse`.
- The current state module imports citation and retrieval result types, but the graph has not yet reused `retrieve_relevant_chunks(...)` or `generate_cited_answer(...)`. That reuse task should stay unchecked until retrieval and answer nodes exist.

### Step 10a: Deterministic Intent Classification

The first behavior node is `classify_intent(...)`. It is shaped like a LangGraph node: it accepts the current agent state and returns an updated state.

For now, classification is deterministic and rule-based. This keeps the support agent usable without a chat-completion model. The node detects policy, troubleshooting, compatibility, comparison, review-summary, clarification, escalation-candidate, and unsupported-category questions. It also extracts product hints from product IDs, titles, brands, categories, and subcategories.

Unsupported categories matter because the demo store knowledge base is scoped to electronics. For example, a question about opened clothes should not retrieve and cite the electronics return policy as if it applies to apparel.

In [ ]:
from support_agent import classify_intent

intent_preview_questions = [
    "What is the return policy?",
    "How do I fix a speaker that will not power on?",
    "Is this case compatible with my model?",
    "Compare these Bluetooth speakers",
    "What do customers say about the Garmin band?",
    "Tell me about mobile accessories",
    "What is the return policy for opened clothes?",
    "Help",
    "Where is my order?",
]

for question in intent_preview_questions:
    classified_state = classify_intent(initial_support_state(question, provider="nebius"))
    print(f"{question} -> {classified_state['intent']}")
    if classified_state.get("product_hints"):
        print("  product hints:", classified_state["product_hints"])
    if classified_state.get("escalation"):
        print("  escalation:", classified_state["escalation"]["reason"])


### Step 10b: Retrieval Planning

The next node is `retrieve_context(...)`. It plans a retrieval query from the detected intent, chooses retrieval settings, calls the existing `retrieve_relevant_chunks(...)` function, and stores the returned chunks back into agent state.

This keeps retrieval logic centralized in `rag_pipeline/retrieval.py`; the support agent only decides how and when to call it.

For clarification and escalation candidates, retrieval is skipped for now because those paths should ask for more information or prepare handoff metadata instead of pretending the knowledge base can answer account-specific questions.

In [ ]:
from support_agent import retrieve_context

policy_state = retrieve_context(
    classify_intent(initial_support_state("What is the return policy?", provider="nebius"))
)

print("Intent:", policy_state["intent"])
print("Retrieval query:", policy_state["retrieval_query"])
print("Result count:", len(policy_state["retrieved_results"]))

for index, result in enumerate(policy_state["retrieved_results"], start=1):
    metadata = result.document.metadata
    print(f"{index}. {metadata.get('doc_type')} | lexical={result.lexical_score} | {metadata.get('source_path')}")

In [ ]:
clarification_state = retrieve_context(
    classify_intent(initial_support_state("Help", provider="nebius"))
)

clarification_state["intent"], clarification_state["retrieval_k"], len(clarification_state["retrieved_results"])

### Step 10c: Context Sufficiency

After retrieval, the agent should not automatically answer. It first checks whether the retrieved context looks strong enough for the detected intent.

`assess_context(...)` currently uses deterministic signals:

- whether retrieval returned results
- the strongest lexical overlap score
- whether retrieved document types match the detected intent

For example, a return-policy question should retrieve a `store_policy` document. If the context is weak or mismatched, the agent should ask for clarification instead of producing a shaky answer. Account/order-specific questions are treated as escalation candidates and skip retrieval for now.

In [ ]:
from support_agent import assess_context

sufficiency_questions = [
    "What is the return policy?",
    "Help",
    "Where is my order?",
]

for question in sufficiency_questions:
    state = assess_context(
        retrieve_context(classify_intent(initial_support_state(question, provider="nebius")))
    )
    print(question)
    print("  intent:", state["intent"])
    print("  retrieved:", len(state["retrieved_results"]))
    print("  has enough context:", state["has_enough_context"])
    print("  needs clarification:", state["needs_clarification"])


### Step 10d: Answer Generation Node

The answer node is `generate_answer(...)`. It uses retrieved chunks already stored in agent state and formats them with the existing deterministic citation logic.

This required a small refactor in `answer_generation.py`: `build_cited_answer_from_results(...)` now formats a cited answer from an existing result list. The older `generate_cited_answer(...)` still works and now calls that reusable formatter after doing retrieval.

That means the LangGraph agent can avoid duplicate retrieval: retrieval happens in `retrieve_context(...)`, then answer formatting happens in `generate_answer(...)`.

In [ ]:
from support_agent import generate_answer

answer_state = generate_answer(
    assess_context(
        retrieve_context(
            classify_intent(initial_support_state("What is the return policy?", provider="nebius"))
        )
    )
)

print("Retrieval query:", answer_state["retrieval_query"])
print(answer_state["answer"])
print("Citations:")
for citation in answer_state["citations"]:
    print(f"[{citation.id}] {citation.label} | {citation.source_path}")

### Step 10e: Clarification Node

The clarification node is `generate_clarification(...)`. It is used when the question is too vague or retrieved context is not strong enough to answer safely.

Clarification responses do not cite sources because they are not grounded answers from retrieved documents. They are follow-up questions that help the user provide enough detail for a later retrieval step.

In [ ]:
from support_agent import generate_clarification

clarification_answer_state = generate_clarification(
    assess_context(
        retrieve_context(
            classify_intent(initial_support_state("Help", provider="nebius"))
        )
    )
)

print("Intent:", clarification_answer_state["intent"])
print("Needs clarification:", clarification_answer_state["needs_clarification"])
print("Answer:", clarification_answer_state["answer"])
print("Citation count:", len(clarification_answer_state["citations"]))

### Step 10f: Escalation Preparation Node

The escalation preparation node is `prepare_escalation(...)`. It does not create a persistent ticket yet. Instead, it prepares handoff metadata and a safe response for questions that need account/order data, order-specific workflows, urgent safety-focused support, or specialist review.

Ticket persistence remains Phase 9. For Phase 7, the important behavior is that the agent recognizes when the local knowledge base should not pretend to answer.

In [ ]:
from support_agent import prepare_escalation

escalation_questions = [
    "Where is my order?",
    "The device smoked and sparked",
]

for question in escalation_questions:
    escalation_state = prepare_escalation(
        assess_context(
            retrieve_context(
                classify_intent(initial_support_state(question, provider="nebius"))
            )
        )
    )
    print(question)
    print("  intent:", escalation_state["intent"])
    print("  reason:", escalation_state["escalation"]["reason"])
    print("  answer:", escalation_state["answer"])


### Step 10g: Graph Assembly

The node functions are now connected with LangGraph in `build_support_agent_graph(...)`.

Current graph:

```text
START -> classify

classify -> retrieve   when the question can use knowledge-base retrieval
classify -> clarify    when the question is too vague
classify -> escalate   when the question needs account/order/safety support

retrieve -> assess

assess -> answer       when retrieved context is strong enough
assess -> clarify      when retrieved context is weak or ambiguous
assess -> escalate     for escalation candidates

answer / clarify / escalate -> END
```

`run_support_agent(...)` compiles and invokes the graph, then returns a `SupportAgentResult` that can later be adapted into the FastAPI `/chat` response.

In [ ]:
from support_agent import run_support_agent

graph_preview_questions = [
    "What is the return policy?",
    "Help",
    "Where is my order?",
]

for question in graph_preview_questions:
    result = run_support_agent(question, provider="nebius")
    print(question)
    print("  intent:", result.intent)
    print("  has enough context:", result.has_enough_context)
    print("  needs clarification:", result.needs_clarification)
    print("  citations:", len(result.citations))
    if result.escalation:
        print("  escalation reason:", result.escalation["reason"])
    print("  answer:", result.answer.splitlines()[0])


### Step 10h: Validation Cases

Phase 7 now has a repeatable validation script: `rag_pipeline/validate_support_agent.py`.

The script checks both layers:

- direct LangGraph agent output, including intent, clarification, and escalation flags
- FastAPI `/chat` output, including status code, citations, and retrieved context

Current validation cases:

- return policy
- product setup
- troubleshooting
- comparison
- underspecified clarification
- order escalation
- unsupported category

Run it from `backend/`:

```powershell
uv run python ../rag_pipeline/validate_support_agent.py
```

In [3]:
# Optional notebook version. This runs the same validation module used from the command line.
from validate_support_agent import main as validate_support_agent

validate_support_agent()

c:\Users\uday0\Documents\GenAIAcademy\Agentic-AI\CSAgent\backend\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


ok: return policy | intent=store_policy | citations=1 | contexts=4
ok: product setup | intent=product_question | citations=4 | contexts=4
ok: troubleshooting | intent=troubleshooting | citations=4 | contexts=4
ok: comparison | intent=comparison | citations=4 | contexts=6
ok: underspecified clarification | intent=clarification_needed | citations=0 | contexts=0
ok: order escalation | intent=escalation_candidate | citations=0 | contexts=0
ok: unsupported category | intent=unsupported_category | citations=0 | contexts=0

Validated 7 support-agent cases.


## Pause and Reflect

Before moving to Phase 8 frontend integration, make sure these ideas are clear:

- We have multiple source types, but one document shape.
- Metadata is separate from searchable text.
- Normalization improves text quality without changing document count.
- Chunking converts documents into smaller retrieval passages.
- Embeddings convert chunk text into vectors that retrieval can compare.
- Chroma stores embedded chunks for local vector search.
- Retrieval finds supporting chunks for a user question.
- Answer generation should cite the retrieved sources instead of making unsupported claims.
- LangGraph makes the support workflow explicit: classify, retrieve, assess, answer, clarify, or escalate.
- Unsupported categories should not cite in-domain policy documents.

Next step: connect the React frontend support chat UI to the LangGraph-backed FastAPI /chat endpoint.